In [197]:
import pandas as pd
import numpy as np

loans_raw = pd.read_csv("data/library_loans_clean.csv", parse_dates=["loan_date", "return_date"])
members_raw = pd.read_csv("data/library_members.csv")

loans = loans_raw.copy()
members = members_raw.copy()

print(loans.shape)
print(members.shape)

(600, 8)
(148, 3)


In [198]:
loans.info()
members.info()

<class 'pandas.DataFrame'>
RangeIndex: 600 entries, 0 to 599
Data columns (total 8 columns):
 #   Column       Non-Null Count  Dtype         
---  ------       --------------  -----         
 0   loan_id      600 non-null    str           
 1   member_id    600 non-null    str           
 2   title        600 non-null    str           
 3   genre        600 non-null    str           
 4   loan_date    597 non-null    datetime64[us]
 5   return_date  564 non-null    datetime64[us]
 6   late_fee     600 non-null    float64       
 7   returned     600 non-null    bool          
dtypes: bool(1), datetime64[us](2), float64(1), str(4)
memory usage: 33.5 KB
<class 'pandas.DataFrame'>
RangeIndex: 148 entries, 0 to 147
Data columns (total 3 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   member_id   148 non-null    str    
 1   branch      136 non-null    str    
 2   birth_year  130 non-null    float64
dtypes: float64(1), str(2)
memory usage: 3

## Part 2: Go deeper

### 1.

- **There are 31 loans where the member_id is not in the member-table which is about 5,2% of all loans**
- **They stand for 347,5:- in late fees which is 4,6% of the total, so nothing out of the ordinary**
- **Their borrowing patterns regarding genre is nothing significant:**
- Children    10
- Sci-fi       5
- Crime        4
- Fantasy      4
- Romance      4
- History      4
- **I see no obvoius reason for why they (6 lenders) should be missing from the member table**

In [199]:
test_merge = loans.merge(members, on='member_id', how='left', indicator=True)

test_merge["_merge"].value_counts()

_merge
both          585
left_only      31
right_only      0
Name: count, dtype: int64

In [200]:
print(31/600)

0.051666666666666666


In [201]:
mask = ~loans['member_id'].isin(members['member_id'])

print(loans.loc[mask, 'late_fee'].sum())
print(loans.loc[mask, 'late_fee'].sum() / loans["late_fee"].sum())

347.5
0.04641068447412354


In [202]:
loans["genre"].value_counts()

genre
Crime       155
Children    130
Sci-fi       89
Fantasy      87
Romance      82
History      57
Name: count, dtype: int64

In [203]:
loans.loc[mask, "genre"].value_counts()

genre
Children    10
Sci-fi       5
Crime        4
Fantasy      4
Romance      4
History      4
Name: count, dtype: int64

In [204]:
loans.loc[mask, "member_id"].unique()

<StringArray>
['M0097', 'M0068', 'M0056', 'M0070', 'M0016', 'M0004']
Length: 6, dtype: str

In [205]:
loans.loc[mask, ["loan_id", 
                 "member_id", 
                 "title", 
                 "genre", 
                 "loan_date", 
                 "return_date", 
                 "late_fee", 
                 "returned"]
                 ].sort_values("member_id", ascending=True).head(31)

,loan_id,member_id,title,genre,loan_date,return_date,late_fee,returned
517,L-30518,M0004,Harbour Lights,Romance,2026-06-03,2026-06-19,0.0,True
514,L-30515,M0004,The Memory Engine,Sci-fi,2026-06-04,2026-07-12,42.5,True
500,L-30501,M0004,A People Remembered,History,2026-05-31,2026-06-28,17.5,True
418,L-30419,M0004,Mimmi and the Moon,Children,2026-05-06,2026-05-18,0.0,True
559,L-30560,M0016,Salt and Iron,History,2026-06-16,2026-07-06,0.0,True
495,L-30496,M0016,Dead Reckoning Lane,Crime,2026-05-29,2026-07-01,30.0,True
124,L-30125,M0016,Between the Lines,Romance,2026-02-10,2026-02-19,0.0,True
376,L-30377,M0016,The Hanseatic Road,History,2026-04-25,2026-05-18,5.0,True
171,L-30172,M0016,The Summer House,Romance,2026-02-24,2026-03-03,0.0,True
576,L-30577,M0016,The Dragon Who Was Afraid,Children,2026-06-21,2026-07-02,0.0,True


### 2.

- **There is only one member that has not borrowed a book**
- **As for preference i think that the merge gives a fuller picture of the data, but if I was to use it in a program
the second version might provide a clearer way to follow the code, and doesnt need to create a new column just for calculation.**

In [206]:
test_merge = loans.merge(members, on='member_id', how='outer', indicator=True)

test_merge["_merge"].value_counts()

_merge
both          585
left_only      31
right_only      1
Name: count, dtype: int64

In [207]:
members_in_loans = members['member_id'].isin(loans['member_id']).sum()
members_total = members['member_id'].count()

print(members_total - members_in_loans)

1


### 3.

There are 143 rows where age at the loan is none, of those 3 lack lone date and 97 lack birth year and 31 could be explained by there beeing no member doing the loan. Other than that the minimum age at loan is also -4 which seems off, as does the max value of 126. If we remove ages under 1 and over 120 we get the result that min age is 16 and max age is 69 instead. More reasonable. The youngest readers seems to be found in the History category. But there is a big but here: The average reader in the genre "children" is 41.1 by the same measure. So i think that these people are better thaught of as "lenders" rather than "readers".

In [208]:
test_merge = loans.merge(members, on='member_id', how='left', indicator=True)

test_merge["_merge"].value_counts()

_merge
both          585
left_only      31
right_only      0
Name: count, dtype: int64

In [209]:
test_merge.info()

<class 'pandas.DataFrame'>
RangeIndex: 616 entries, 0 to 615
Data columns (total 11 columns):
 #   Column       Non-Null Count  Dtype         
---  ------       --------------  -----         
 0   loan_id      616 non-null    str           
 1   member_id    616 non-null    str           
 2   title        616 non-null    str           
 3   genre        616 non-null    str           
 4   loan_date    613 non-null    datetime64[us]
 5   return_date  579 non-null    datetime64[us]
 6   late_fee     616 non-null    float64       
 7   returned     616 non-null    bool          
 8   branch       539 non-null    str           
 9   birth_year   519 non-null    float64       
 10  _merge       616 non-null    category      
dtypes: bool(1), category(1), datetime64[us](2), float64(2), str(5)
memory usage: 44.8 KB


In [210]:
print(test_merge["loan_id"].duplicated().value_counts())

loan_id
False    600
True      16
Name: count, dtype: int64


In [211]:
mask = test_merge["loan_id"].duplicated()

test_merge.loc[mask].sort_values("loan_id").head(32)

,loan_id,member_id,title,genre,loan_date,return_date,late_fee,returned,branch,birth_year,_merge
30,L-30030,M0071,Blue Moon Protocol,Sci-fi,2026-01-14,2026-01-19,0.0,True,Central,1970.0,both
61,L-30060,M0071,The Locksmith,Crime,2026-01-22,2026-02-02,0.0,True,Central,1970.0,both
145,L-30143,M0065,The Glass Mage,Fantasy,2026-02-15,2026-03-24,40.0,True,Central,1988.0,both
170,L-30167,M0033,Forest Friends,Children,2026-02-23,2026-03-12,0.0,True,South,1975.0,both
172,L-30168,M0033,The Dragon Who Was Afraid,Children,2026-02-23,2026-03-30,35.0,True,South,1975.0,both
289,L-30284,M0015,The Hanseatic Road,History,2026-03-29,2026-04-22,7.5,True,Central,1995.0,both
310,L-30304,M0033,A Second Spring,Romance,2026-04-03,2026-05-10,40.0,True,South,1975.0,both
371,L-30364,M0015,Night Train to Nowhere,Crime,2026-04-21,2026-04-25,0.0,True,Central,1995.0,both
417,L-30409,M0065,The Brave Little Boat,Children,2026-05-04,2026-05-30,12.5,True,Central,1988.0,both
466,L-30457,M0033,The Ember Crown,Fantasy,2026-05-17,NaT,0.0,False,South,1975.0,both


In [212]:
test_merge = test_merge.drop_duplicates(subset="loan_id")

In [213]:
print(test_merge["loan_id"].duplicated().value_counts())

loan_id
False    600
Name: count, dtype: int64


In [214]:
test_merge["age_at_the_loan"] = test_merge["loan_date"].dt.year - test_merge["birth_year"]

test_merge["age_at_the_loan"].describe().round(2)

count    500.00
mean      42.37
std       24.38
min       -4.00
25%       30.00
50%       39.00
75%       48.00
max      126.00
Name: age_at_the_loan, dtype: float64

In [215]:
print((test_merge["age_at_the_loan"] < 1).value_counts())

age_at_the_loan
False    585
True      15
Name: count, dtype: int64


In [216]:
print((test_merge["age_at_the_loan"] > 120).value_counts())

age_at_the_loan
False    572
True      28
Name: count, dtype: int64


In [217]:
test_merge["age_at_the_loan"] = test_merge["age_at_the_loan"].where(
    test_merge["age_at_the_loan"].between(1, 120), 
    np.nan)

In [218]:
test_merge["age_at_the_loan"].describe().round(2)

count    457.00
mean      38.77
std       11.72
min       16.00
25%       30.00
50%       39.00
75%       47.00
max       69.00
Name: age_at_the_loan, dtype: float64

In [219]:
print(test_merge["birth_year"].isna().value_counts())

birth_year
False    503
True      97
Name: count, dtype: int64


In [220]:
print(test_merge["loan_date"].isna().value_counts())

loan_date
False    597
True       3
Name: count, dtype: int64


In [221]:
print(test_merge["age_at_the_loan"].isna().value_counts())

age_at_the_loan
False    457
True     143
Name: count, dtype: int64


In [222]:
test_merge.groupby("genre")["age_at_the_loan"].agg(["count", "mean"]).round(1)

,count,mean
genre,,
Children,101,41.1
Crime,120,38.4
Fantasy,67,39.8
History,45,35.2
Romance,58,38.1
Sci-fi,66,37.8


### 4.

#### Birthyear anomalies
- There where 18 members with missing birth year. Plus 8 members with their birthyear before 1906 (120+ years old) and 3 with their birthyear before 2026 (less than 0 years old)
- (Im writing this in november 2026)
- The missing birthyears stays at NaN(not a number) as we can't guess the birthyear and the impossible values are turned to NaN for the same reason.

#### Missing branch
- There are 12 members with missing info on branch.
- These are changed to "Unknown" so that they are not excluded when we do statistics per branch.

#### Duplicates
- There are 4 duplicated members, and thay seem not to be related to the 6 unknown lenders below.
- These rows are removed from the dataset as they are full duplicates and no data is lost.
- Index is reset upon removal. New number of members = 144

#### 6 Unknown lenders
- There where 31 loans where the members where not in the member table, these where all created by 6 lenders.
- When looking at these loans compared to other loans there was no indication that the loans themselves were unusual, so this is likley a problem having to do with the members table (as explained in 2.1).
- These member_id's are now added to the members table with branch="Unknown" and no value in birth_year.
- This makes the new total number of members 150, and the members with missing branch and missing birth_year goes up by 6, respectivly. 



In [223]:
members["birth_year"].describe().round(2)

count     130.00
mean     1983.38
std        25.16
min      1900.00
25%      1978.25
50%      1988.00
75%      1996.75
max      2030.00
Name: birth_year, dtype: float64

In [224]:
print(members["birth_year"].isna().value_counts())

birth_year
False    130
True      18
Name: count, dtype: int64


In [225]:
print((members["birth_year"] < 1906).value_counts())

birth_year
False    140
True       8
Name: count, dtype: int64


In [226]:
print((members["birth_year"] > 2026).value_counts())

birth_year
False    145
True       3
Name: count, dtype: int64


In [227]:
members["birth_year"] = members["birth_year"].where(
    members["birth_year"].between(1906, 2026), 
    np.nan)

members["birth_year"].describe().round(2)

count     119.00
mean     1987.82
std        12.06
min      1957.00
25%      1979.00
50%      1988.00
75%      1996.50
max      2010.00
Name: birth_year, dtype: float64

In [228]:
members["branch"].describe()

count         136
unique          5
top       Central
freq           40
Name: branch, dtype: object

In [229]:
members["branch"].value_counts()

branch
Central    40
South      34
North      32
East       17
Harbour    13
Name: count, dtype: int64

In [230]:
members["branch"].unique()

<StringArray>
['Central', 'North', nan, 'South', 'East', 'Harbour']
Length: 6, dtype: str

In [231]:
members["branch"].isna().sum()

np.int64(12)

In [232]:
members["branch"] = members["branch"].fillna("Unknown")

print(members["branch"].isna().sum())
members["branch"].value_counts()

0


branch
Central    40
South      34
North      32
East       17
Harbour    13
Unknown    12
Name: count, dtype: int64

In [233]:
mask = members.duplicated(keep=False)

members.loc[mask].sort_values("member_id").head(15)

,member_id,branch,birth_year
13,M0015,Central,1995.0
14,M0015,Central,1995.0
31,M0033,South,1975.0
32,M0033,South,1975.0
63,M0065,Central,1988.0
64,M0065,Central,1988.0
68,M0071,Central,1970.0
69,M0071,Central,1970.0


In [234]:
loans_test = loans_raw.copy()

mask = ~loans_test["member_id"].isin(members["member_id"])

loans_test.loc[mask, "member_id"].unique()

<StringArray>
['M0097', 'M0068', 'M0056', 'M0070', 'M0016', 'M0004']
Length: 6, dtype: str

In [235]:
members = members.drop_duplicates().reset_index(drop=True)

print(members.shape)
print(members["member_id"].is_unique)

(144, 3)
True


In [236]:
new_members = pd.DataFrame({
    "member_id": ['M0097', 'M0068', 'M0056', 'M0070', 'M0016', 'M0004'],
    "branch": ["Unknown", "Unknown", "Unknown", "Unknown", "Unknown", "Unknown"],
    "birth_year": [np.nan, np.nan, np.nan, np.nan, np.nan, np.nan]
})


members = pd.concat([members, new_members], ignore_index=True)

In [237]:
members = members.sort_values("member_id").reset_index(drop=True)

In [238]:
members.sort_values("member_id").head(16)

,member_id,branch,birth_year
0,M0001,Central,1971.0
1,M0002,North,1986.0
2,M0003,North,NaN
3,M0004,Unknown,NaN
4,M0005,Unknown,2001.0
5,M0006,South,NaN
6,M0007,Central,1961.0
7,M0008,East,NaN
8,M0009,North,1957.0
9,M0010,Unknown,1987.0


In [239]:
print(members.shape)
print(members["member_id"].is_unique)

(150, 3)
True
